In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("cardio_cleaned.csv")
df.head()

,age_group,height_group,weight_group,ap_hi_group,ap_lo_group,gender,cholesterol,gluc,smoke,alco,active,cardio
0,3,1,2,2,1,1,2,2,0,0,1,0
1,0,2,3,1,1,1,1,1,0,0,1,1
2,2,2,2,1,1,1,1,1,0,0,1,0
3,0,2,4,1,1,2,1,1,1,1,1,0
4,3,1,2,1,1,1,1,1,0,0,1,0


In [3]:
y = df["cardio"]
X = df.drop(columns=["cardio"])

In [4]:
depths = list(range(1, 21))          # testowane głębokości 1..20
seeds  = list(range(10))             # 10 różnych random_state

In [5]:
rows = []

for seed in seeds:
    # 70/15/15 split
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )
    for d in depths:
        pipe = Pipeline(steps=[
            ("clf", DecisionTreeClassifier(max_depth=d, random_state=seed))
        ])
        pipe.fit(X_train, y_train)
        acc_tr = accuracy_score(y_train, pipe.predict(X_train))
        acc_val = accuracy_score(y_val, pipe.predict(X_val))
        rows.append({
            "seed": seed,
            "max_depth": d,
            "train_acc": acc_tr,
            "val_acc": acc_val
        })
results = pd.DataFrame(rows)

In [6]:
# Średnie i odchylenia po seedach
summary = (
    results.groupby("max_depth")[["train_acc", "val_acc"]]
    .agg(["mean", "std"])
)
# Porządniejsze nazwy kolumn
summary.columns = ["train_mean", "train_std", "val_mean", "val_std"]
summary = summary.reset_index()

display(summary)

,max_depth,train_mean,train_std,val_mean,val_std
0,1,0.710232,0.000581,0.709406,0.001804
1,2,0.710232,0.000581,0.709406,0.001804
2,3,0.721269,0.001096,0.718773,0.002525
3,4,0.725551,0.001422,0.723156,0.004398
4,5,0.729927,0.001474,0.727972,0.004583
5,6,0.731510,0.001362,0.727745,0.005541
6,7,0.733557,0.001206,0.727539,0.004255
7,8,0.736024,0.001222,0.726051,0.004645
8,9,0.738876,0.001177,0.724604,0.004529
9,10,0.742396,0.001094,0.723264,0.003647


In [7]:
# Wybór najlepszej głębokości po ŚREDNIM train accuracy (tie -> mniejsze depth)
best_row = summary.sort_values(
    by=["train_mean", "max_depth"], ascending=[False, True]
).iloc[0]
best_depth = int(best_row["max_depth"])
print(f"Najlepsza głębokość po średnim TRAIN accuracy: {best_depth} "
      f"(train_mean={best_row['train_mean']:.4f}, val_mean={best_row['val_mean']:.4f})")

Najlepsza głębokość po średnim TRAIN accuracy: 20 (train_mean=0.7877, val_mean=0.6943)


In [8]:
# Wybór najlepszej głębokości po ŚREDNIM val accuracy (tie -> mniejsze depth)
best_row = summary.sort_values(
    by=["val_mean", "max_depth"], ascending=[False, True]
).iloc[0]
best_depth = int(best_row["max_depth"])
print(f"Najlepsza głębokość po średnim VAL accuracy: {best_depth} "
      f"(val_mean={best_row['val_mean']:.4f}, train_mean={best_row['train_mean']:.4f})")

Najlepsza głębokość po średnim VAL accuracy: 5 (val_mean=0.7280, train_mean=0.7299)


In [9]:
# Dla wybranej głębokości (na podstawie walidacji) policzmy też średni TEST accuracy (po seedach)
test_accs = []
for seed in seeds:
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )
    pipe = Pipeline(steps=[
        ("clf", DecisionTreeClassifier(max_depth=best_depth, random_state=seed))
    ])
    pipe.fit(X_train, y_train)
    test_accs.append(accuracy_score(y_test, pipe.predict(X_test)))

print(f"TEST (best depth={best_depth}): {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")

TEST (best depth=5): 0.7296 ± 0.0031
